<a href="https://www.kaggle.com/code/shamanthakreddymallu/s6e7-lookup-model?scriptVersionId=332801763" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Empirical Lookup Model — S6E7 Student Health Risk

The structural probe found the label generator is (approximately):
- `unhealthy = (high stress) AND (sleep < 6)`
- `fit = (low stress) AND (active) AND (sleep >= 7)`
- else `at-risk`, with ~0.5-1% label noise and a narrow jitter band at each cliff

For a rule+noise generator, the Bayes-optimal probability estimator is a **frequency table**: P(class | stress, activity, sleep-bin) computed directly from train counts. This notebook builds that table with proper fold discipline and emits OOF/test probability artifacts in the same format as the GBM/NN notebooks, so it can slot straight into the blend.

CPU-only, runs in a few minutes.

In [ ]:
# PATHS
TRAIN_PATH    = '/kaggle/input/competitions/playground-series-s6e7/train.csv'
TEST_PATH     = '/kaggle/input/competitions/playground-series-s6e7/test.csv'
SAMPLE_PATH   = '/kaggle/input/competitions/playground-series-s6e7/sample_submission.csv'
ORIGINAL_PATH = '/kaggle/input/datasets/ziya07/college-student-health-behavior-dataset/student_health_dataset_50k.csv'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (balanced_accuracy_score, confusion_matrix,
                             classification_report)
from itertools import product
import warnings
warnings.filterwarnings('ignore')

TARGET   = 'health_condition'
CAT_COLS = ['diet_type', 'stress_level', 'sleep_quality',
            'physical_activity_level', 'smoking_alcohol', 'gender']
NUM_COLS = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
            'step_count', 'exercise_duration', 'water_intake']

train    = pd.read_csv(TRAIN_PATH)
test     = pd.read_csv(TEST_PATH)
original = pd.read_csv(ORIGINAL_PATH)

for df in [train, test, original]:
    for c in CAT_COLS:
        df[c] = df[c].astype(str).str.strip().str.lower().replace('nan', np.nan)

original[TARGET] = original[TARGET].astype(str).str.strip().str.lower()
original = original.drop(columns=['student_id', 'timestamp'])

print(f'Train {train.shape} | Test {test.shape} | Original {original.shape}')

## Lookup Key Construction

Key = (stress_level × physical_activity_level × sleep_bin), all with explicit 'missing' levels. Sleep binned at 0.05 inside the two jitter bands (5.8–6.3 and 6.8–7.3) where resolution matters, and 0.25 elsewhere — keeps bins well-populated without sacrificing cliff resolution.

In [ ]:
# variable-resolution sleep bin edges: fine inside jitter bands, coarse elsewhere
edges = np.concatenate([
    np.arange(3.0, 5.8, 0.25),
    np.arange(5.8, 6.3, 0.05),    # unhealthy cliff band
    np.arange(6.3, 6.8, 0.25),
    np.arange(6.8, 7.3, 0.05),    # fit cliff band
    np.arange(7.3, 10.26, 0.25),
])
edges = np.unique(edges.round(3))
print(f'{len(edges)-1} sleep bins')

def make_key(df):
    stress   = df['stress_level'].fillna('missing')
    activity = df['physical_activity_level'].fillna('missing')
    sleep_bin = pd.cut(df['sleep_duration'], bins=edges).astype(str)
    sleep_bin = sleep_bin.where(df['sleep_duration'].notna(), 'missing')
    return stress + '|' + activity + '|' + sleep_bin

train['key']    = make_key(train)
test['key']     = make_key(test)
original['key'] = make_key(original)

print(f"Unique keys in train: {train['key'].nunique():,}")
print(f"Test keys not present in train: {(~test['key'].isin(set(train['key']))).sum():,}")

## Setup — same folds and protocol as every other model notebook

In [ ]:
USE_ORIGINAL = True
train['source']    = 'kaggle'
original['source'] = 'original'
train_augmented = pd.concat(
    [train[['key', TARGET, 'source']],
     original[['key', TARGET, 'source']]],
    ignore_index=True
)

le    = LabelEncoder()
y_aug = le.fit_transform(train_augmented[TARGET])
print('Class encoding:', dict(zip(le.classes_, range(3))))

N_FOLDS, SEED, N_CLASSES = 5, 42, 3
kag_idx  = np.where(train_augmented['source'] == 'kaggle')[0]
orig_idx = np.where(train_augmented['source'] == 'original')[0]
y_kag    = y_aug[kag_idx]

skf         = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLD_SPLITS = list(skf.split(kag_idx, y_aug[kag_idx]))

def optimize_class_weights(probs, y_true, coarse=31, refine=25):
    grid       = np.exp(np.linspace(np.log(0.5), np.log(40), coarse))
    best_score = balanced_accuracy_score(y_true, probs.argmax(1))
    best_w     = np.ones(3)
    for wf, wu in product(grid, grid):
        w = np.array([1.0, wf, wu])
        s = balanced_accuracy_score(y_true, (probs * w).argmax(1))
        if s > best_score:
            best_score, best_w = s, w
    if refine:
        ff = np.linspace(best_w[1] * 0.8, best_w[1] * 1.2, refine)
        fu = np.linspace(best_w[2] * 0.8, best_w[2] * 1.2, refine)
        for wf, wu in product(ff, fu):
            w = np.array([1.0, wf, wu])
            s = balanced_accuracy_score(y_true, (probs * w).argmax(1))
            if s > best_score:
                best_score, best_w = s, w
    return best_w, best_score

## The Lookup 'Model'

Per fold: count classes per key on (train-fold + original), convert to probabilities with additive smoothing toward the (stress × activity) cell marginal, then look up val-fold / test rows. Unseen keys fall back to the cell marginal, then the global prior.

Smoothing: `P = (counts + m * cell_prior) / (n + m)` with m=20 pseudo-counts — thin bins get pulled toward their cell distribution instead of screaming 0/1.

In [ ]:
M_SMOOTH = 20.0

def build_and_apply(train_keys, train_y, query_keys):
    dfc = pd.DataFrame({'key': train_keys, 'y': train_y})
    dfc['cell'] = dfc['key'].str.rsplit('|', n=1).str[0]

    # per-key class counts
    key_counts = dfc.groupby(['key', 'y']).size().unstack(fill_value=0)
    key_counts = key_counts.reindex(columns=range(N_CLASSES), fill_value=0)

    # per-cell marginal distributions
    cell_counts = dfc.groupby(['cell', 'y']).size().unstack(fill_value=0)
    cell_counts = cell_counts.reindex(columns=range(N_CLASSES), fill_value=0)
    cell_prior  = cell_counts.div(cell_counts.sum(axis=1), axis=0)

    global_prior = np.bincount(train_y, minlength=N_CLASSES) / len(train_y)

    # smoothed key probabilities
    key_cell = pd.Series(key_counts.index.str.rsplit('|', n=1).str[0],
                         index=key_counts.index)
    prior_for_key = cell_prior.reindex(key_cell.values).values.copy()
    nan_rows = np.isnan(prior_for_key).any(axis=1)
    prior_for_key[nan_rows] = global_prior
    n_key = key_counts.values.sum(axis=1, keepdims=True)
    key_probs = (key_counts.values + M_SMOOTH * prior_for_key) / (n_key + M_SMOOTH)
    key_prob_df = pd.DataFrame(key_probs, index=key_counts.index,
                               columns=range(N_CLASSES))

    # ---- lookup for queries ----
    q = pd.DataFrame({'key': query_keys})
    q['cell'] = q['key'].str.rsplit('|', n=1).str[0]

    out = key_prob_df.reindex(q['key'].values).values.copy()
    miss = np.isnan(out).any(axis=1)
    if miss.any():
        cell_fill = cell_prior.reindex(q.loc[miss, 'cell'].values).values.copy()
        out[miss] = cell_fill
        still = np.isnan(out).any(axis=1)
        if still.any():
            out[still] = global_prior
    return out

oof_probs  = np.zeros((len(kag_idx), N_CLASSES))
test_probs = np.zeros((len(test), N_CLASSES))
fold_scores = []

aug_keys = train_augmented['key'].values

for fold, (tr_kag, val_kag) in enumerate(FOLD_SPLITS, 1):
    tr_idx  = np.concatenate([kag_idx[tr_kag], orig_idx])
    val_idx = kag_idx[val_kag]

    val_p = build_and_apply(aug_keys[tr_idx], y_aug[tr_idx], aug_keys[val_idx])
    oof_probs[val_kag] = val_p
    test_probs += build_and_apply(aug_keys[tr_idx], y_aug[tr_idx],
                                  test['key'].values) / N_FOLDS

    s = balanced_accuracy_score(y_aug[val_idx], val_p.argmax(1))
    fold_scores.append(s)
    print(f'Fold {fold}: balanced_acc = {s:.5f}')

raw_oof_score = balanced_accuracy_score(y_kag, oof_probs.argmax(1))
print(f'\nRaw OOF (argmax): {raw_oof_score:.5f}')
print(f'CV mean         : {np.mean(fold_scores):.5f} +/- {np.std(fold_scores):.5f}')

np.save('oof_lookup.npy', oof_probs)
np.save('test_lookup.npy', test_probs)

# Decision-Rule Optimization on OOF

In [ ]:
global_w, opt_oof_score = optimize_class_weights(oof_probs, y_kag)
print(f'Raw  OOF balanced accuracy : {raw_oof_score:.5f}')
print(f'Optimized weights          : {dict(zip(le.classes_, global_w.round(4)))}')
print(f'Opt. OOF balanced accuracy : {opt_oof_score:.5f}   (+{opt_oof_score - raw_oof_score:.5f})')

# Post-Modeling Analysis

In [ ]:
probs_w   = oof_probs * global_w
preds_kag = probs_kag = probs_w.argmax(1)

print(classification_report(y_kag, preds_kag, target_names=le.classes_))

train_kag = train.reset_index(drop=True)
stress_missing_tr = train_kag['stress_level'].isna().values
sleep_missing_tr  = train_kag['sleep_duration'].isna().values

for name, mask in [('stress present', ~stress_missing_tr),
                   ('stress MISSING', stress_missing_tr),
                   ('sleep MISSING',  sleep_missing_tr),
                   ('both present',   ~stress_missing_tr & ~sleep_missing_tr)]:
    s = balanced_accuracy_score(y_kag[mask], preds_kag[mask])
    print(f'{name:16s}: balanced_acc = {s:.5f}   (n={mask.sum():,})')

cm_pct = confusion_matrix(y_kag, preds_kag, normalize='true') * 100
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax, linewidths=0.5)
ax.set_title('Lookup Model — Confusion (row %)', fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.show()

## Jitter-Band Check

The whole point of the lookup: better-calibrated probabilities inside the cliff bands. This compares its band-only balanced accuracy so the blend result is interpretable.

In [ ]:
sleep = train_kag['sleep_duration']
band_mask = (sleep.between(5.8, 6.3) | sleep.between(6.8, 7.3)).fillna(False).values

for name, mask in [('inside jitter bands', band_mask),
                   ('outside bands (sleep present)', ~band_mask & sleep.notna().values)]:
    s = balanced_accuracy_score(y_kag[mask], preds_kag[mask])
    print(f'{name:32s}: balanced_acc = {s:.5f}   (n={mask.sum():,})')

# Submission (standalone — the blend is where this artifact really goes)

In [ ]:
test_w = test_probs * global_w
labels = le.inverse_transform(test_w.argmax(1))

sub = pd.read_csv(SAMPLE_PATH)
sub[TARGET] = labels
sub.to_csv('submission.csv', index=False)
print('Saved: submission.csv (+ oof_lookup.npy / test_lookup.npy for blending)')
print('\nPredicted class distribution (test):')
print(pd.Series(labels).value_counts().to_string())
print('\nTrain distribution for reference:')
print(train[TARGET].value_counts().to_string())

**Next step:** add to the blend notebook's `ARTIFACTS` dict:
```python
'lookup': (f'{NB_BASE}/s6e7-lookup-model/oof_lookup.npy',
           f'{NB_BASE}/s6e7-lookup-model/test_lookup.npy'),
```
Watch (a) the correlation heatmap — expect low correlation with GBMs specifically on band rows, (b) the all-subsets table — GBM+lookup pairs vs GBM alone is the verdict on whether the discovered structure adds anything the trees hadn't priced in.